In [1]:
import pandas as pd
import numpy as np

In [2]:
orthoDist = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")
display(orthoDist)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC
Human ID,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,1.130272,0.0000
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,1.141118,0.1250
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,1.211158,0.0000
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.814870,0.0000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.959842,0.1875
...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
groupedHumanGene = orthoDist.groupby("Gene stable ID")["Mouse gene stable ID"].apply(list).reset_index(name="Mouse gene stable ID")
groupedHumanGene = groupedHumanGene.merge(orthoDist.loc[:, ["Gene stable ID", "Mouse homology type"]])
manyToOneHuman = groupedHumanGene[groupedHumanGene["Mouse gene stable ID"].apply(len) > 1]
groupedHumanGene["Duplicated Species"] = np.where((groupedHumanGene["Gene stable ID"].isin(manyToOneHuman["Gene stable ID"])) & (groupedHumanGene["Mouse homology type"] == "ortholog_one2many"), "Human", "NA")

In [4]:
groupedHumanGene

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Duplicated Species
0,ENSG00000000003,[ENSMUSG00000067377],ortholog_one2one,NA
1,ENSG00000000005,[ENSMUSG00000031250],ortholog_one2one,NA
2,ENSG00000000419,[ENSMUSG00000078919],ortholog_one2one,NA
3,ENSG00000000457,[ENSMUSG00000026584],ortholog_one2one,NA
4,ENSG00000000460,[ENSMUSG00000041406],ortholog_one2one,NA
...,...,...,...,...
28009,ENSG00000310576,[ENSMUSG00000035595],ortholog_one2one,NA
28010,ENSG00000310579,[nan],NaN,NA
28011,ENSG00000310583,[nan],NaN,NA
28012,ENSG00000310590,[nan],NaN,NA


In [5]:
orthoDistHumanDup = orthoDist.merge(groupedHumanGene.loc[:, ["Gene stable ID", "Duplicated Species"]], left_on="Gene stable ID", right_on="Gene stable ID", how="outer")

In [6]:
groupedMouseGene = orthoDist.groupby("Mouse gene stable ID")["Gene stable ID"].apply(list).reset_index(name="Gene stable ID")
groupedMouseGene = groupedMouseGene.merge(orthoDist.loc[:, ["Mouse gene stable ID", "Mouse homology type"]])
manyToOneMouse = groupedMouseGene[groupedMouseGene["Gene stable ID"].apply(len) > 1]
groupedMouseGene["Duplicated Species"] = np.where((groupedMouseGene["Mouse gene stable ID"].isin(manyToOneMouse["Mouse gene stable ID"])) & (groupedMouseGene["Mouse homology type"] == "ortholog_one2many"), "Mouse", "NA")

In [7]:
groupedMouseGene[groupedMouseGene["Duplicated Species"] != "NA"]

,Mouse gene stable ID,Gene stable ID,Mouse homology type,Duplicated Species
73,ENSMUSG00000000308,"[ENSG00000223572, ENSG00000237289]",ortholog_one2many,Mouse
74,ENSMUSG00000000308,"[ENSG00000223572, ENSG00000237289]",ortholog_one2many,Mouse
157,ENSMUSG00000000804,"[ENSG00000129204, ENSG00000170832]",ortholog_one2many,Mouse
158,ENSMUSG00000000804,"[ENSG00000129204, ENSG00000170832]",ortholog_one2many,Mouse
184,ENSMUSG00000000982,"[ENSG00000275385, ENSG00000276085, ENSG0000027...",ortholog_one2many,Mouse
...,...,...,...,...
22358,ENSMUSG00000116652,"[ENSG00000286025, ENSG00000286102, ENSG0000028...",ortholog_one2many,Mouse
22427,ENSMUSG00000121607,"[ENSG00000146938, ENSG00000165246]",ortholog_one2many,Mouse
22428,ENSMUSG00000121607,"[ENSG00000146938, ENSG00000165246]",ortholog_one2many,Mouse
22458,ENSMUSG00000144229,"[ENSG00000288841, ENSG00000289282]",ortholog_one2many,Mouse


In [8]:
orthoDistHumanMouseDup = orthoDistHumanDup.merge(groupedMouseGene.loc[:, ["Mouse gene stable ID", "Duplicated Species"]], on="Mouse gene stable ID", how="outer")
orthoDistHumanMouseDup["Duplicated Species"] = np.where(orthoDistHumanMouseDup["Duplicated Species_x"] == "NA", orthoDistHumanMouseDup["Duplicated Species_y"], orthoDistHumanMouseDup["Duplicated Species_x"])

In [9]:
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, ~orthoDistHumanMouseDup.columns.isin(["Duplicated Species_x", "Duplicated Species_y"])].drop_duplicates()
display(orthoDistHumanMouseDup)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC,Duplicated Species
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.151442,0.0000,NA
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.001638,0.1875,NA
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.161786,0.0625,NA
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.000003,0.2500,NA
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.661562,0.0000,NA
...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
orthoDistHumanMouseDup.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.csv", index=False)
orthoDistHumanMouseDup.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.parquet", index=False)